## **Tính định lý Bayes bằng tay**

In [13]:
# -*- coding: utf-8 -*-
"""
Bước 1: Tính toán công thức Bayes thủ công (Không sử dụng thư viện).
"""


def tinh_xac_suat_bayes(p_prior, p_likelihood, p_likelihood_neg):
    """
    Tính và trả về xác suất hậu nghiệm P(A|B).

    p_prior          : Xác suất tiền nghiệm P(A)
    p_likelihood     : Khả năng P(B | A)
    p_likelihood_neg : Khả năng ngược P(B | không A)
    """
    tu = p_likelihood * p_prior
    mau = tu + p_likelihood_neg * (1.0 - p_prior)
    return tu / mau


print("Kiem tra Y te: Du doan kha nang mac benh")
print("  Tyle benh nen: 1%, Do chinh xac xet nghiem: 99%.")
p_kq = tinh_xac_suat_bayes(p_prior=0.01, p_likelihood=0.99, p_likelihood_neg=0.01)
print(f"  Ket qua Duong tinh -> Xac suat mac benh thuc te = {p_kq:.4f}")
print()

print("Khao sat su thay doi theo ty le benh nen:")
print("  Ty le benh        Xac suat hau nghiem")
for rate in [0.001, 0.01, 0.05, 0.10, 0.50]:
    print(f"    {rate:6.3f}              {tinh_xac_suat_bayes(rate, 0.99, 0.01):.4f}")
print()

print("Bai toan Quan ly Chat luong: Phan tich san pham loi")
print("  Xuong A: 80% san luong, ty le phe pham 1%.")
print("  Xuong B: 20% san luong, ty le phe pham 5%.")
p_xuong_b = tinh_xac_suat_bayes(p_prior=0.20, p_likelihood=0.05, p_likelihood_neg=0.01)
print(f"  Lay ngau nhien 1 phe pham -> Kha nang thuoc Xuong B = {p_xuong_b:.4f}")
print()

print("Mo phong thong ke tren quy mo 1000 san pham:")
so_loi_a = 800 * 0.01
so_loi_b = 200 * 0.05
print(f"  Xuong A: 800 san pham -> {so_loi_a:.0f} phe pham")
print(f"  Xuong B: 200 san pham -> {so_loi_b:.0f} phe pham")
tong_phe_pham = so_loi_a + so_loi_b
print(f"  Ty le den tu Xuong B = {so_loi_b} / {tong_phe_pham} = {so_loi_b / tong_phe_pham:.4f}")

Kiem tra Y te: Du doan kha nang mac benh
  Tyle benh nen: 1%, Do chinh xac xet nghiem: 99%.
  Ket qua Duong tinh -> Xac suat mac benh thuc te = 0.5000

Khao sat su thay doi theo ty le benh nen:
  Ty le benh        Xac suat hau nghiem
     0.001              0.0902
     0.010              0.5000
     0.050              0.8390
     0.100              0.9167
     0.500              0.9900

Bai toan Quan ly Chat luong: Phan tich san pham loi
  Xuong A: 80% san luong, ty le phe pham 1%.
  Xuong B: 20% san luong, ty le phe pham 5%.
  Lay ngau nhien 1 phe pham -> Kha nang thuoc Xuong B = 0.5556

Mo phong thong ke tren quy mo 1000 san pham:
  Xuong A: 800 san pham -> 8 phe pham
  Xuong B: 200 san pham -> 10 phe pham
  Ty le den tu Xuong B = 10.0 / 18.0 = 0.5556


## **Làm bằng scikit‐learn**

In [14]:
# -*- coding: utf-8 -*-
"""
Bước 2: Nạp tập dữ liệu tin nhắn và chuyển đổi văn bản sang dạng ma trận số.
"""

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# Nạp dữ liệu từ file csv
data_frame = pd.read_csv("tin_nhan.csv")

# Kiểm tra các cột trong file
print("Danh sach cac cot trong file:", data_frame.columns.tolist())

# Tìm cột chứa nhãn (chấp nhận la_rac, label, is_spam, ...)
label_col = [c for c in data_frame.columns if c.strip().lower() in ['la_rac', 'label', 'is_spam', 'spam', 'nhan']][0]
content_col = [c for c in data_frame.columns if c != label_col][0]

print("Kich thuoc ma tran du lieu:", data_frame.shape)
print()
print("Phan bo so luong theo tung phan loai:")
print(data_frame[label_col].value_counts())
print()
print("Ty le tin nhan rac trong tap du lieu:", round(data_frame[label_col].mean(), 4))
print()

print("Moi xem 3 tin nhan dau tien:")
for idx in range(min(3, len(data_frame))):
    print(f"  [{data_frame[label_col].iloc[idx]}] {data_frame[content_col].iloc[idx]}")
print()

# Tách từ theo khoảng trắng bằng regex
bow_vectorizer = CountVectorizer(token_pattern=r"(?u)\S+")
X_matrix = bow_vectorizer.fit_transform(data_frame[content_col])

print("Kich thuoc ma tran tan suat tu:", X_matrix.shape)
print("Tong so tu trong kho tu dien:", len(bow_vectorizer.vocabulary_))
print()

# Thống kê tổng lượt xuất hiện của từng từ theo 2 nhóm
df_counts = pd.DataFrame(X_matrix.toarray(), columns=bow_vectorizer.get_feature_names_out())
df_counts[label_col] = data_frame[label_col].values
df_sum = df_counts.groupby(label_col).sum().T
df_sum.columns = ["thuong", "rac"]

print("Top 10 tu xuat hien pho bien nhat trong tin RAC:")
print(df_sum.sort_values("rac", ascending=False).head(10))
print()
print("Top 10 tu xuat hien pho bien nhat trong tin THUONG:")
print(df_sum.sort_values("thuong", ascending=False).head(10))

Danh sach cac cot trong file: ['noi_dung', 'la_rac']
Kich thuoc ma tran du lieu: (300, 2)

Phan bo so luong theo tung phan loai:
la_rac
0    180
1    120
Name: count, dtype: int64

Ty le tin nhan rac trong tap du lieu: 0.4

Moi xem 3 tin nhan dau tien:
  [0] cáo gửi cáo thầy nhóm cáo gửi có cho
  [1] miễn giảm ngay hạn miễn tay nhé thưởng cho nhanh
  [0] cho bạn mai gửi nhóm bài chiều

Kich thuoc ma tran tan suat tu: (300, 40)
Tong so tu trong kho tu dien: 40

Top 10 tu xuat hien pho bien nhat trong tin RAC:
       thuong  rac
tay         3   48
ngay        4   47
nhanh       9   44
chót        6   43
giảm        6   42
lúc        39   42
quà         8   41
click       5   40
phí         6   40
miễn        4   39

Top 10 tu xuat hien pho bien nhat trong tin THUONG:
       thuong  rac
thầy       74    7
bài        73    4
cáo        72    8
chiều      68    8
ạ          65    7
tài        64   11
tập        63    6
gửi        61   10
lớp        60    2
nhóm       58    7


## **Huấn luyện bộ lọc thư rác và chấm điểm**

In [15]:
# -*- coding: utf-8 -*-
"""
Bước 3: Huấn luyện thuật toán phân loại tin nhắn rác và đánh giá hiệu năng.
"""

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

# Nạp dữ liệu và tiến hành chuyển đổi
data_frame = pd.read_csv("tin_nhan.csv")
bow_extractor = CountVectorizer(token_pattern=r"(?u)\S+")
X_features = bow_extractor.fit_transform(data_frame["noi_dung"])
y_labels = data_frame["la_rac"]

# Phân chia tập dữ liệu Train / Test
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_labels, test_size=0.25, random_state=17, stratify=y_labels
)
print("So luong mau huan luyen:", X_train.shape[0])
print("So luong mau danh gia   :", X_test.shape[0])
print()

# Khởi tạo và huấn luyện mô hình Naive Bayes
nb_model = MultinomialNB(alpha=1.0)
nb_model.fit(X_train, y_train)
y_predictions = nb_model.predict(X_test)

# Đánh giá bằng ma trận nhầm lẫn (Confusion Matrix)
tn, fp, fn, tp = confusion_matrix(y_test, y_predictions).ravel()
print("Kiet qua Ma tran nham lan:")
print(f"  TN = {tn:2d}   tin thuong phan loai chinh xac")
print(f"  FP = {fp:2d}   tin thuong bi nham thanh rac")
print(f"  FN = {fn:2d}   tin rac bi bo xot")
print(f"  TP = {tp:2d}   tin rac bi phat hien va chan")
print()

print(f"Do chinh xac tong the (Accuracy)  = {accuracy_score(y_test, y_predictions):.4f}")
print(f"Do chinh xac tin rac (Precision)  = {precision_score(y_test, y_predictions):.4f}")
print(f"Tyle phat hien rac    (Recall)     = {recall_score(y_test, y_predictions):.4f}")
print()

# Kiểm thử mô hình với các câu tin nhắn ngẫu nhiên
custom_messages = [
    "trung thuong khuyen mai nhanh tay click ngay",
    "hop nhom bao cao chieu mai phong 305",
    "ban gui giup em tai lieu bai tap nhe",
]
X_custom = bow_extractor.transform(custom_messages)
prob_spam = nb_model.predict_proba(X_custom)[:, 1]

print("Kiem thu du doan tren tin nhan moi:")
for msg, prob in zip(custom_messages, prob_spam):
    tag = "RAC" if prob >= 0.5 else "thuong"
    print(f"  [{tag:6s}] Xac suat = {prob:.4f}  <-- {msg}")

So luong mau huan luyen: 225
So luong mau danh gia   : 75

Kiet qua Ma tran nham lan:
  TN = 45   tin thuong phan loai chinh xac
  FP =  0   tin thuong bi nham thanh rac
  FN =  4   tin rac bi bo xot
  TP = 26   tin rac bi phat hien va chan

Do chinh xac tong the (Accuracy)  = 0.9467
Do chinh xac tin rac (Precision)  = 1.0000
Tyle phat hien rac    (Recall)     = 0.8667

Kiem thu du doan tren tin nhan moi:
  [RAC   ] Xac suat = 0.9992  <-- trung thuong khuyen mai nhanh tay click ngay
  [thuong] Xac suat = 0.0983  <-- hop nhom bao cao chieu mai phong 305
  [thuong] Xac suat = 0.0719  <-- ban gui giup em tai lieu bai tap nhe


## **Tự cài đặt để thấy tận mắt**

In [16]:
# -*- coding: utf-8 -*-
"""
Bước 4: Tự viết thuật toán Naive Bayes thủ công để minh họa vấn đề Zero Frequency.

Viết trực tiếp bằng Python thuần, không dùng thư viện scikit-learn để
thấy rõ tác dụng của kỹ thuật làm trơn Laplace.
"""

from collections import Counter

# Tập dữ liệu mẫu dạng thu nhỏ
du_lieu_mau = [
    ("trung thuong ngay", 1),
    ("khuyen mai giam gia", 1),
    ("trung thuong click", 1),
    ("hop nhom chieu mai", 0),
    ("bai tap nhom nop mai", 0),
    ("thay gui tai lieu", 0),
]

danh_sach_tu = sorted({word for text, _ in du_lieu_mau for word in text.split()})
V_size = len(danh_sach_tu)
print(f"Kho tu dien bao gom {V_size} tu:")
# Định dạng in 9 từ trên mỗi hàng
for idx in range(0, V_size, 9):
    print("  " + " ".join(danh_sach_tu[idx:idx + 9]))
print()

# Thống kê tần suất từ xuất hiện theo từng nhóm nhãn
tan_suat = {0: Counter(), 1: Counter()}
tong_tu_lop = {0: 0, 1: 0}
for text, label in du_lieu_mau:
    for word in text.split():
        tan_suat[label][word] += 1
        tong_tu_lop[label] += 1

print("Thong ke tan suat tu trong tung nhom:")
print(f"  {'Tu don':12s} {'Rac(1)':>6s} {'Thuong(0)':>9s}")
for word in danh_sach_tu:
    print(f"  {word:12s} {tan_suat[1][word]:6d} {tan_suat[0][word]:9d}")
print()


def tinh_p_tu_theo_lop(word, label, alpha_param):
    """Tính xác suất P(word | label) kết hợp hệ số làm trơn Laplace (alpha_param)."""
    return (tan_suat[label][word] + alpha_param) / (tong_tu_lop[label] + alpha_param * V_size)


def tinh_diem_xac_suat(text, label, alpha_param):
    """Tính xác suất tích lũy của văn bản theo nhãn lớp."""
    prob_score = 0.5  # Xác suất tiền nghiệm bằng nhau giữa 2 lớp P(Spam) = P(Ham) = 0.5
    for word in text.split():
        if word in danh_sach_tu:
            prob_score *= tinh_p_tu_theo_lop(word, label, alpha_param)
    return prob_score


cau_danh_gia = "trung thuong bai tap"
print(f'Thu nghiem voi cau: "{cau_danh_gia}"')
print("Tu 'bai' chi co o nhom thuong, hoan toan vang mat o nhom rac.")
print()

for alpha_val in [0.0, 1.0]:
    score_rac = tinh_diem_xac_suat(cau_danh_gia, 1, alpha_val)
    score_thuong = tinh_diem_xac_suat(cau_danh_gia, 0, alpha_val)
    ket_luan = "rac" if score_rac > score_thuong else "thuong"
    print(f"alpha = {alpha_val}")
    print(f"  Diem nhom rac    = {score_rac:.10f}")
    print(f"  Diem nhom thuong = {score_thuong:.10f}")
    print(f"  Ket luan cuoi    = {ket_luan}")
    print()

print("Khi alpha = 0.0: Ca 2 diem deu bang 0 khien mo hinh bi te liet.")
print("Nhom rac bang 0 do chua bao gio xuat hien tu 'bai'.")
print("Nhom thuong bang 0 do chua bao gio xuat hien tu 'trung'.")
print("Khi alpha = 1.0 (Laplace Smoothing): Loai bo xac suat 0 va mo hinh ra ket qua dung.")

Kho tu dien bao gom 18 tu:
  bai chieu click gia giam gui hop khuyen lieu
  mai ngay nhom nop tai tap thay thuong trung

Thong ke tan suat tu trong tung nhom:
  Tu don       Rac(1) Thuong(0)
  bai               0         1
  chieu             0         1
  click             1         0
  gia               1         0
  giam              1         0
  gui               0         1
  hop               0         1
  khuyen            1         0
  lieu              0         1
  mai               1         2
  ngay              1         0
  nhom              0         2
  nop               0         1
  tai               0         1
  tap               0         1
  thay              0         1
  thuong            2         0
  trung             2         0

Thu nghiem voi cau: "trung thuong bai tap"
Tu 'bai' chi co o nhom thuong, hoan toan vang mat o nhom rac.

alpha = 0.0
  Diem nhom rac    = 0.0000000000
  Diem nhom thuong = 0.0000000000
  Ket luan cuoi    = thuong

alpha = 1.0
  Die

## **Naive Bayes cho số liệu liên tục, đặt cạnh hồi quy logistic.**

In [17]:
# -*- coding: utf-8 -*-
"""
Bước 5: Áp dụng Gaussian Naive Bayes cho đặc trưng liên tục và so sánh hiệu năng với Logistic Regression.
"""

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

# Đọc tập dữ liệu điểm số và thời gian học của sinh viên
data_frame = pd.read_csv("sinh_vien.csv")
X_features = data_frame[["gio_on", "diem_giua_ky"]]
y_targets = data_frame["qua_mon"]

# Chia dữ liệu thành tập huấn luyện và tập kiểm thử
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_targets, test_size=0.25, random_state=17, stratify=y_targets
)

# Đào tạo và so sánh độ chính xác của hai thuật toán
for model_name, classifier in [("Gaussian Naive Bayes", GaussianNB()),
                               ("Hoi quy Logistic    ", LogisticRegression())]:
    classifier.fit(X_train, y_train)
    accuracy_val = accuracy_score(y_test, classifier.predict(X_test))
    print(f"{model_name}: Do chinh xac (Accuracy) = {accuracy_val:.4f}")
print()

# Khởi tạo mô hình Gaussian Naive Bayes riêng biệt để trích xuất tham số
gnb_model = GaussianNB().fit(X_train, y_train)

# theta_ biểu thị giá trị trung bình (mean), var_ biểu thị phương sai (variance)
print("Cac tham số phan phoi Gaussian hoc duoc:")
print(f"  {'':16s} {'gio_on':>10s} {'diem_giua_ky':>14s}")
for idx, label_desc in enumerate(["Nhom 0 (Rot) ", "Nhom 1 (Dau) "]):
    mean_vals = gnb_model.theta_[idx]
    std_devs = gnb_model.var_[idx] ** 0.5
    print(f"  {label_desc} Trung binh {mean_vals[0]:10.2f} {mean_vals[1]:14.2f}")
    print(f"  {'':12s} Do lech    {std_devs[0]:10.2f} {std_devs[1]:14.2f}")
print()

priors_str = ", ".join(f"{float(val):.4f}" for val in gnb_model.class_prior_)
print(f"Xac suat tien nghiem theo phan loai trong tap Train: {priors_str}")
print()

# Dự đoán xác suất cho các trường hợp mẫu thử mới
new_cases = pd.DataFrame({"gio_on": [5, 13, 20, 28],
                          "diem_giua_ky": [4.0, 6.0, 7.0, 9.0]})
prob_pass = gnb_model.predict_proba(new_cases)[:, 1]

print("Du doan xac suat qua mon cho cac sinh vien moi:")
print(f"  {'Gio on':>8s} {'Diem GK':>9s} {'P(Pass)':>12s}")
for (_, row_data), p_val in zip(new_cases.iterrows(), prob_pass):
    print(f"  {row_data.gio_on:8.0f} {row_data.diem_giua_ky:9.1f} {p_val:12.4f}")

Gaussian Naive Bayes: Do chinh xac (Accuracy) = 0.9000
Hoi quy Logistic    : Do chinh xac (Accuracy) = 0.9000

Cac tham số phan phoi Gaussian hoc duoc:
                       gio_on   diem_giua_ky
  Nhom 0 (Rot)  Trung binh       8.18           3.99
               Do lech          5.29           1.92
  Nhom 1 (Dau)  Trung binh      20.03           6.70
               Do lech          5.81           1.50

Xac suat tien nghiem theo phan loai trong tap Train: 0.4111, 0.5889

Du doan xac suat qua mon cho cac sinh vien moi:
    Gio on   Diem GK      P(Pass)
         5       4.0       0.0137
        13       6.0       0.6550
        20       7.0       0.9855
        28       9.0       0.9999
